# 12a — Validation Evaluation

This notebook validates the **saved fitted Week 5 models** on the held-out validation datasets.

It is intentionally simple:

- loads the saved `.joblib` models from `models/`;
- does **not** rebuild the Week 5 pipeline;
- does **not** refit or retune models;
- uses `predict()` and `predict_proba()` directly;
- saves validation metrics, predictions, probabilities, confusion matrices, and summary tables;
- does not require Matplotlib.


## 1. Setup

In [ ]:
from pathlib import Path
import re
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Run this notebook from the project repository.")

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
TABLES_DIR = OUTPUTS_DIR / "tables"

for folder in [METRICS_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET = "label"
ID_COLUMN = "patient_id"

print("Project root:", PROJECT_ROOT)
print("Models folder:", MODELS_DIR)


## 2. Find the saved Week 5 models

In [ ]:
model_files = sorted(MODELS_DIR.glob("*.joblib"))

if not model_files:
    raise FileNotFoundError(
        "No .joblib models were found in models/. "
        "Copy the Week 5 saved models into that folder and rerun."
    )

print(f"Found {len(model_files)} model(s):")
for path in model_files:
    print(" -", path.name)


## 3. Build model inventory

In [ ]:
def clean_text(value):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).strip().lower(),
    ).strip("_")


MODEL_SUFFIXES = {
    "logistic_regression": "Logistic Regression",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
}

DATASET_PREFIXES = {
    "demographics_plus_questionnaire": "Demographics + Questionnaire",
    "wearable_plus_questionnaire": "Wearable + Questionnaire",
    "multimodal": "Full Multimodal",
    "full_multimodal": "Full Multimodal",
}


records = []

for path in model_files:
    stem = clean_text(path.stem)

    model_name = None
    dataset_name = None

    for suffix, display_name in MODEL_SUFFIXES.items():
        if stem.endswith(suffix):
            model_name = display_name
            dataset_key = stem[: -(len(suffix) + 1)]
            dataset_name = DATASET_PREFIXES.get(dataset_key)
            break

    if model_name and dataset_name:
        records.append({
            "dataset": dataset_name,
            "model": model_name,
            "filename": path.name,
            "model_path": path,
        })
    else:
        print("Skipped unrecognized file:", path.name)


model_inventory = pd.DataFrame(records)

if model_inventory.empty:
    raise RuntimeError("No recognized Week 5 model files were found.")

display(model_inventory)


## 4. Locate validation datasets

If automatic discovery does not find the correct file, set it manually in `MANUAL_VALIDATION_FILES`.


In [ ]:
MANUAL_VALIDATION_FILES = {
    # "Demographics + Questionnaire": DATA_DIR / "processed" / "YOUR_FILE.csv",
    # "Wearable + Questionnaire": DATA_DIR / "processed" / "YOUR_FILE.csv",
    # "Full Multimodal": DATA_DIR / "processed" / "YOUR_FILE.csv",
}


all_csvs = list(DATA_DIR.rglob("*.csv")) if DATA_DIR.exists() else []


def find_validation_file(dataset_name):
    if dataset_name in MANUAL_VALIDATION_FILES:
        path = Path(MANUAL_VALIDATION_FILES[dataset_name])
        return path if path.exists() else None

    aliases = {
        "Demographics + Questionnaire": [
            "demographics_questionnaire",
            "demographics_plus_questionnaire",
        ],
        "Wearable + Questionnaire": [
            "wearable_questionnaire",
            "wearable_plus_questionnaire",
        ],
        "Full Multimodal": [
            "multimodal_full",
            "full_multimodal",
            "multimodal",
        ],
    }[dataset_name]

    candidates = []

    for path in all_csvs:
        name = clean_text(path.stem)

        if not any(x in name for x in ["validation", "valid", "_val"]):
            continue

        score = max(
            sum(part in name for part in clean_text(alias).split("_"))
            for alias in aliases
        )

        if score > 0:
            candidates.append((score, path))

    if not candidates:
        return None

    return sorted(
        candidates,
        key=lambda x: x[0],
        reverse=True,
    )[0][1]


validation_files = {
    dataset: find_validation_file(dataset)
    for dataset in model_inventory["dataset"].unique()
}

display(pd.DataFrame([
    {
        "dataset": dataset,
        "validation_file": str(path) if path else None,
    }
    for dataset, path in validation_files.items()
]))


## 5. Evaluate saved models

In [ ]:
def metric_values(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "precision_macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
    }


results = []
prediction_rows = []
probability_rows = []
audit_rows = []


for _, row in model_inventory.iterrows():
    dataset = row["dataset"]
    model_name = row["model"]
    validation_file = validation_files.get(dataset)

    if validation_file is None:
        print(f"✗ {dataset} | {model_name}: validation file not found")
        continue

    try:
        model = joblib.load(row["model_path"])
        validation_df = pd.read_csv(validation_file)

        if TARGET not in validation_df.columns:
            raise ValueError(f"Missing target column: {TARGET}")

        y_true = validation_df[TARGET].copy()

        if hasattr(model, "feature_names_in_"):
            feature_columns = list(model.feature_names_in_)
        else:
            feature_columns = [
                c for c in validation_df.columns
                if c not in {TARGET, ID_COLUMN}
            ]

        missing = [
            c for c in feature_columns
            if c not in validation_df.columns
        ]

        if missing:
            raise ValueError(
                f"Validation data are missing model features: {missing[:10]}"
            )

        X_val = validation_df[feature_columns].copy()

        # No fit() call — these are the fitted Week 5 models.
        y_pred = model.predict(X_val)

        metrics = metric_values(y_true, y_pred)

        results.append({
            "dataset": dataset,
            "model": model_name,
            "artifact_filename": row["filename"],
            "n_validation": len(y_true),
            **metrics,
        })

        ids = (
            validation_df[ID_COLUMN].astype(str).values
            if ID_COLUMN in validation_df.columns
            else np.arange(len(validation_df)).astype(str)
        )

        for pid, true_label, pred_label in zip(ids, y_true, y_pred):
            prediction_rows.append({
                ID_COLUMN: pid,
                "dataset": dataset,
                "model": model_name,
                "true_label": true_label,
                "predicted_label": pred_label,
            })

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_val)

            classes = getattr(model, "classes_", None)

            if classes is None and hasattr(model, "named_steps"):
                classifier = model.named_steps.get("classifier")
                classes = getattr(
                    classifier,
                    "classes_",
                    np.arange(proba.shape[1]),
                )

            for i, (pid, true_label) in enumerate(zip(ids, y_true)):
                item = {
                    ID_COLUMN: pid,
                    "dataset": dataset,
                    "model": model_name,
                    "true_label": true_label,
                }

                for j, class_label in enumerate(classes):
                    item[f"prob_class_{class_label}"] = proba[i, j]

                probability_rows.append(item)

        cm = confusion_matrix(y_true, y_pred)
        pd.DataFrame(cm).to_csv(
            METRICS_DIR
            / f"{clean_text(dataset)}__{clean_text(model_name)}_validation_confusion_matrix.csv",
            index=False,
        )

        steps = (
            " -> ".join(name for name, _ in model.steps)
            if hasattr(model, "steps")
            else type(model).__name__
        )

        audit_rows.append({
            "dataset": dataset,
            "model": model_name,
            "filename": row["filename"],
            "pipeline_steps": steps,
            "status": "LOADED",
        })

        print(
            f"✓ {dataset} | {model_name} | "
            f"Macro F1={metrics['macro_f1']:.4f}"
        )

    except Exception as exc:
        print(
            f"✗ {dataset} | {model_name}: "
            f"{type(exc).__name__}: {exc}"
        )


## 6. Save validation outputs

In [ ]:
validation_results = pd.DataFrame(results)

if validation_results.empty:
    raise RuntimeError("No model completed validation successfully.")

validation_results = validation_results.sort_values(
    ["macro_f1", "balanced_accuracy"],
    ascending=[False, False],
).reset_index(drop=True)

validation_results.insert(
    0,
    "rank",
    np.arange(1, len(validation_results) + 1),
)

validation_predictions = pd.DataFrame(prediction_rows)
validation_probabilities = pd.DataFrame(probability_rows)
artifact_audit = pd.DataFrame(audit_rows)

validation_results.to_csv(
    METRICS_DIR / "validation_results.csv",
    index=False,
)

validation_predictions.to_csv(
    METRICS_DIR / "validation_predictions.csv",
    index=False,
)

validation_probabilities.to_csv(
    METRICS_DIR / "validation_probabilities.csv",
    index=False,
)

artifact_audit.to_csv(
    TABLES_DIR / "week5_model_artifact_audit.csv",
    index=False,
)

validation_results.to_csv(
    TABLES_DIR / "validation_comparison.csv",
    index=False,
)

best_models = (
    validation_results
    .sort_values(
        ["dataset", "macro_f1", "balanced_accuracy"],
        ascending=[True, False, False],
    )
    .groupby("dataset", as_index=False)
    .first()
)

best_models.to_csv(
    TABLES_DIR / "validation_best_models.csv",
    index=False,
)

print("Saved validation outputs successfully.")
display(validation_results)
display(best_models)


## 7. Final check

A successful run means:

- saved Week 5 models were loaded from `models/`;
- no model was refitted;
- predictions and probabilities were generated on held-out validation data;
- validation outputs were saved to `outputs/metrics/` and `outputs/tables/`.
